# 🔷 Módulo 10 - Notebook 02: Agregación espacial y resoluciones H3

## 📊 Análisis jerárquico multiescala

**Libro:** Saliendo de lo Pandito  
**Módulo:** 10 - Indexación Hexagonal Uber H3  
**Duración estimada:** 70 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Agregar** datos por hexágonos H3  
✅ **Comparar** múltiples resoluciones  
✅ **Implementar** jerarquías padre-hijo  
✅ **Optimizar** resolución según caso de uso  
✅ **Visualizar** densidades a diferentes escalas

---

## 📋 Pre-requisitos

* ✅ Notebook 10_01 completado (Introducción a H3)
* ✅ Conocimiento de h3.geo_to_h3()
* ✅ Familiaridad con groupby y agregaciones

---

## 📚 Contenido

1. Agregación por Hexágonos
2. Comparación de Resoluciones
3. Jerarquías H3 (parent/children)
4. Selección de Resolución Óptima
5. Zoom In/Out Dinámico
6. Caso Integrador: Análisis Multiescala de Ventas

---

## 💡 Por qué importa

**Agregación jerárquica transforma el análisis:**

* 🔍 **Zoom:** Del continente al edificio sin recalcular
* ⚡ **Velocidad:** Agregaciones instantáneas
* 📊 **Flexibilidad:** Mismos datos, múltiples vistas
* 🎯 **Precisión:** Elegir granularidad según necesidad

**El poder de H3 está en su jerarquía**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Verificar que tengamos coordenadas
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros: {len(df_ventas):,}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   🏪 Sucursales: {df_ventas['sucursal_id'].nunique()}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n🗺️ Coordenadas disponibles:")
    print(f"   • lat: {df_ventas['lat'].min():.4f} a {df_ventas['lat'].max():.4f}")
    print(f"   • lon: {df_ventas['lon'].min():.4f} a {df_ventas['lon'].max():.4f}")
    
    print(f"\n🎯 Este notebook agregará ventas por hexágonos H3")
    print(f"   Se compararán múltiples resoluciones")
    
    print(f"\n📊 Métricas disponibles:")
    print(f"   • Ventas totales: ${df_ventas['ventas'].sum():,.2f}")
    print(f"   • Ventas promedio: ${df_ventas['ventas'].mean():,.2f}")
    print(f"   • Registros por sucursal: {len(df_ventas) / df_ventas['sucursal_id'].nunique():.0f}")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Agregación Jerárquica H3: Multiescala

### 📦 Agregación por Hexágonos

**Flujo básico:**

```python
import h3

# 1. Convertir cada fila a H3
df['h3_index'] = df.apply(
    lambda row: h3.geo_to_h3(row['lat'], row['lon'], resolution=9),
    axis=1
)

# 2. Agregar por hexágono
hex_agg = df.groupby('h3_index').agg({
    'ventas': 'sum',
    'clientes': 'count'
}).reset_index()

# 3. Cada fila = un hexágono con sus métricas
```

---

### 🔷 Comparación de Resoluciones

**Mismo dataset, diferentes granularidades:**

| Resolución | Tamaño Hex | Uso | # Hexágonos |
|------------|------------|-----|-------------|
| **7** | ~5 km² | Ciudad completa | ~10 |
| **9** | ~0.1 km² | Barrios | ~500 |
| **11** | ~0.002 km² | Cuadras | ~25,000 |

**Trade-off:**
* **Resolución baja:** Menos hexágonos, visión general
* **Resolución alta:** Más hexágonos, detalle fino

---

### 📊 Jerarquía Padre-Hijo

**H3 es jerárquico:** Cada hexágono tiene padres (mayor) e hijos (menor).

```python
import h3

hex_id = '89a8100c54fffff'  # Resolución 9

# Obtener padre (resolución 7 = más grande)
padre = h3.h3_to_parent(hex_id, res=7)

# Obtener hijos (resolución 11 = más pequeños)
hijos = h3.h3_to_children(hex_id, res=11)
print(f"Hijos: {len(hijos)}")  # ~49 hexágonos
```

**Visualización:**
```
     [Padre res 7]
    ┌─────────────┐
    │   [Hex res 9] │
    │  ┌───────┐ │
    │  │ 49 hijos │ │
    │  │  res 11  │ │
    │  └───────┘ │
    └─────────────┘
```

---

### 🎯 Selección de Resolución Óptima

**Pregunta:** ¿Qué resolución usar?

**Criterios:**

1. **Área de estudio:**
   * Ciudad: res 7-9
   * Barrio: res 9-11
   * Edificio: res 11-13

2. **Granularidad de datos:**
   * Muchos puntos (millones): res alta (9-11)
   * Pocos puntos (miles): res baja (7-8)

3. **Propósito:**
   * Visualización ejecutiva: res baja
   * Análisis operativo: res alta

---

### 🔍 Zoom In/Out Dinámico

**Caso de uso:** Dashboard interactivo

```python
# Inicio: Vista de ciudad (res 7)
city_view = df.groupby(
    df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 7), axis=1)
)['ventas'].sum()

# Usuario hace zoom: Vista de barrio (res 9)
barrio_view = df.groupby(
    df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)
)['ventas'].sum()

# Usuario hace más zoom: Vista de cuadra (res 11)
cuadra_view = df.groupby(
    df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 11), axis=1)
)['ventas'].sum()
```

**Ventaja:** Mismo dataset, múltiples vistas instantáneas.

---

### 💼 Caso de Uso Empresarial: Densidad de Ventas

**Pregunta:** ¿Dónde están mis mejores zonas?

**Proceso:**

1. **Convertir ventas a H3 (res 9)**
2. **Agregar ventas por hexágono**
3. **Identificar top 10% hexágonos**
4. **Visualizar en mapa**

**Resultado:** Zonas rojas = alta densidad, zonas azules = baja densidad.

---

### ⚡ Ventajas de Agregación H3

✅ **Velocidad:** groupby por índice (string) es instantáneo  
✅ **Flexibilidad:** Cambiar resolución = recalcular en segundos  
✅ **Comparabilidad:** Hexágonos uniformes (vs cuadrículas distorsionadas)  
✅ **Escalabilidad:** De millones de puntos a cientos de hexágonos

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("📊 AGREGACIÓN ESPACIAL JERÁRQUICA H3")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import h3
    print(f"Versión de H3: {h3.__version__}")
except ImportError:
    print("⚠️  H3 no instalado. Ejecuta: %pip install h3")

print("\n🎯 En este notebook aprenderás:")
print("  • Agregar datos por hexágonos H3")
print("  • Comparar múltiples resoluciones")
print("  • h3.h3_to_parent() - Jerarquía")
print("  • h3.h3_to_children() - Subdivisión")

print("\n📖 Métodos clave:")
print("  - df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], res), axis=1)")
print("  - df.groupby('h3_index')['metric'].sum()")
print("  - h3.h3_to_parent(hex_id, res)")
print("  - h3.h3_to_children(hex_id, res)")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 📊 Agregación H3 con datos reales de Los Andes Market

### 🎯 Agregando ventas por hexágono

El dataset ya tiene tres columnas H3 pre-calculadas. Podemos agregar ventas por cada resolución para obtener vistas multiescala:

```python
# Resolución 7 (~1.22km): zonas comerciales
agg_res7 = df.groupby('h3_res7')['ventas'].sum()

# Resolución 8 (~461m): agrupación de sucursales
agg_res8 = df.groupby('h3_res8')['ventas'].sum()

# Resolución 9 (~174m): sucursal individual
agg_res9 = df.groupby('h3_index')['ventas'].sum()
```

---

### 🔄 Zoom in/out con jerarquía H3

```python
# De res 8 a res 7 (zoom out)
df['h3_padre'] = df['h3_res8'].apply(lambda x: h3.h3_to_parent(x, 7))

# De res 9 a res 7 (zoom out 2 niveles)
df['h3_padre7'] = df['h3_index'].apply(lambda x: h3.h3_to_parent(x, 7))
```

---

### 💡 Preguntas de negocio
* ¿Qué resolución muestra la mejor distribución de ventas?
* ¿Hay hexágonos con múltiples sucursales (alta densidad comercial)?
* ¿Cómo cambia el ranking de zonas entre res 7, 8 y 9?

In [0]:
import pandas as pd
import h3
import plotly.express as px

print("📊 AGREGACIÓN H3 CON DATOS REALES DE LOS ANDES MARKET")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None:
    print("\n1️⃣  AGREGACIÓN POR RESOLUCIÓN 7 (~1.22km): Zonas")
    print("-"*70)

    if 'h3_res7' in df_ventas.columns:
        agg_res7 = df_ventas.groupby('h3_res7').agg(
            ventas_total=('ventas', 'sum'),
            ventas_prom=('ventas', 'mean'),
            sucursales=('sucursal_id', 'nunique'),
            registros=('ventas', 'count')
        ).reset_index().sort_values('ventas_total', ascending=False)

        print(f"\n   Hexágonos únicos (res 7): {len(agg_res7)}")
        print("\n   Top 5 zonas por ventas:")
        print(agg_res7.head().round(0))

    print("\n" + "="*70)
    print("\n2️⃣  AGREGACIÓN POR RESOLUCIÓN 8 (~461m): Intermedia")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        agg_res8 = df_ventas.groupby('h3_res8').agg(
            ventas_total=('ventas', 'sum'),
            sucursales=('sucursal_id', 'nunique'),
            registros=('ventas', 'count')
        ).reset_index().sort_values('ventas_total', ascending=False)

        print(f"\n   Hexágonos únicos (res 8): {len(agg_res8)}")
        print("\n   Top 5 zonas por ventas:")
        print(agg_res8.head().round(0))
        print(f"\n   💡 Res 8 agrupa sucursales más cercanas que res 7")

    print("\n" + "="*70)
    print("\n3️⃣  AGREGACIÓN POR RESOLUCIÓN 9 (~174m): Sucursal")
    print("-"*70)

    if 'h3_index' in df_ventas.columns:
        agg_res9 = df_ventas.groupby('h3_index').agg(
            ventas_total=('ventas', 'sum'),
            sucursales=('sucursal_id', 'nunique'),
            zona=('zona', 'first')
        ).reset_index().sort_values('ventas_total', ascending=False)

        print(f"\n   Hexágonos únicos (res 9): {len(agg_res9)}")
        print("\n   Top 5 sucursales por ventas:")
        print(agg_res9.head().round(0))

    print("\n" + "="*70)
    print("\n4️⃣  COMPARACIÓN MULTIESCALA")
    print("-"*70)

    resumen = pd.DataFrame({
        'Resolución': ['Res 7 (~1.22km)', 'Res 8 (~461m)', 'Res 9 (~174m)'],
        'Hexágonos': [len(agg_res7), len(agg_res8), len(agg_res9)],
        'Ventas promedio': [agg_res7['ventas_total'].mean(), agg_res8['ventas_total'].mean(), agg_res9['ventas_total'].mean()],
    })
    print("\n   Comparación de resoluciones:")
    print(resumen.round(0))
    print("\n   💡 Menor resolución = menos hexágonos, más ventas por hex")

    print("\n" + "="*70)
    print("\n5️⃣  ZOOM IN/OUT: Jerarquía padre-hijo")
    print("-"*70)

    # De res 9 → res 7 (zoom out)
    if 'h3_index' in df_ventas.columns:
        df_ventas['h3_padre7'] = df_ventas['h3_index'].apply(lambda x: h3.h3_to_parent(x, 7))
        agg_padre7 = df_ventas.groupby('h3_padre7')['ventas'].sum()

        print(f"\n   Zoom out: Res 9 → Res 7")
        print(f"   Hexágonos res 9: {df_ventas['h3_index'].nunique()}")
        print(f"   Hexágonos res 7 (padre): {df_ventas['h3_padre7'].nunique()}")
        print("\n   💡 Mismos datos, diferente granularidad sin recalcular")

    print("\n" + "="*70)
    print("\n6️⃣  SCATTER: Ventas por hexágono (res 8)")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        # Obtener centro de cada hexágono
        agg_res8['lat'] = agg_res8['h3_res8'].apply(lambda x: h3.h3_to_geo(x)[0])
        agg_res8['lon'] = agg_res8['h3_res8'].apply(lambda x: h3.h3_to_geo(x)[1])

        fig = px.scatter(
            agg_res8, x='lon', y='lat',
            size='ventas_total', color='ventas_total',
            title='🔷 Densidad de Ventas por Hexágono H3 (Res 8)',
            labels={'ventas_total': 'Ventas ($)', 'lat': 'Latitud', 'lon': 'Longitud'},
            template='plotly_white', size_max=30,
            color_continuous_scale='YlOrRd'
        )
        fig.show()
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# 🎯 OPCIONAL: Agregaciones jerárquicas con datos reales

# Descomentar para usar datos reales:
"""
import pandas as pd
import h3

print("💾 Cargando datos con H3 desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    print(f"✅ Datos cargados: {len(df_ventas):,} registros")
    
    print(f"\n📈 Agregaciones Jerárquicas por Resolución H3:")
    
    # Resolución 9 (~174m) - Más granular
    agg_res9 = df_ventas.groupby('h3_index').agg({
        'ventas': ['sum', 'mean', 'count'],
        'sucursal_id': 'nunique'
    }).reset_index()
    agg_res9.columns = ['h3_id', 'ventas_totales', 'ventas_promedio', 'registros', 'num_sucursales']
    
    print(f"\n1️⃣ Resolución 9 (~174m):")
    print(f"   Hexágonos únicos: {len(agg_res9)}")
    print(f"   Ventas totales: ${agg_res9['ventas_totales'].sum():,.2f}")
    display(agg_res9.head())
    
    # Resolución 8 (~461m) - Intermedia
    agg_res8 = df_ventas.groupby('h3_res8').agg({
        'ventas': ['sum', 'mean', 'count'],
        'sucursal_id': 'nunique'
    }).reset_index()
    agg_res8.columns = ['h3_id', 'ventas_totales', 'ventas_promedio', 'registros', 'num_sucursales']
    
    print(f"\n2️⃣ Resolución 8 (~461m):")
    print(f"   Hexágonos únicos: {len(agg_res8)}")
    print(f"   Ventas totales: ${agg_res8['ventas_totales'].sum():,.2f}")
    display(agg_res8.head())
    
    # Resolución 7 (~1.22km) - Más agregado
    agg_res7 = df_ventas.groupby('h3_res7').agg({
        'ventas': ['sum', 'mean', 'count'],
        'sucursal_id': 'nunique'
    }).reset_index()
    agg_res7.columns = ['h3_id', 'ventas_totales', 'ventas_promedio', 'registros', 'num_sucursales']
    
    print(f"\n3️⃣ Resolución 7 (~1.22km):")
    print(f"   Hexágonos únicos: {len(agg_res7)}")
    print(f"   Ventas totales: ${agg_res7['ventas_totales'].sum():,.2f}")
    display(agg_res7.head())
    
    print(f"\n💡 Observación:")
    print(f"   A menor resolución (res 7), menos hexágonos pero más datos agregados.")
    print(f"   A mayor resolución (res 9), más hexágonos pero menos datos por hex.")
    
    print(f"\n🕸️ Comparación de Resoluciones:")
    comparacion = pd.DataFrame({
        'Resolución': [7, 8, 9],
        'Tamaño Aprox': ['~1.22 km', '~461 m', '~174 m'],
        'Hexágonos Únicos': [len(agg_res7), len(agg_res8), len(agg_res9)],
        'Ventas Promedio/Hex': [
            f"${agg_res7['ventas_totales'].mean():,.2f}",
            f"${agg_res8['ventas_totales'].mean():,.2f}",
            f"${agg_res9['ventas_totales'].mean():,.2f}"
        ]
    })
    display(comparacion)
    
    print(f"\n💡 Variables disponibles:")
    print("   • agg_res9: Agregaciones a resolución 9")
    print("   • agg_res8: Agregaciones a resolución 8")
    print("   • agg_res7: Agregaciones a resolución 7")
    
    print(f"\n🎯 Casos de Uso por Resolución:")
    print("   • Res 9: Análisis hiper-local, ubicación precisa de puntos de venta")
    print("   • Res 8: Zonificación de áreas de influencia, rutas de distribución")
    print("   • Res 7: Planificación estratégica, expansión regional")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del notebook 10_02

### ✅ Lo que aprendiste

1. **Agregación por hexágonos H3:**
   - `df['h3_index'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], res), axis=1)`
   - `df.groupby('h3_index')['ventas'].sum()` — agregación instantánea
   - Cada hexágono recibe un ID único (string) que sirve como clave de groupby

2. **Comparación de resoluciones:**
   - Res 7 (~5 km²): vista de ciudad, pocos hexágonos, visión general
   - Res 9 (~0.1 km²): vista de barrio, balance granularidad vs agregación
   - Res 11 (~0.002 km²): vista de cuadra, máximo detalle, muchos hexágonos
   - Trade-off: menos hexágonos = más datos por hexágono, menos detalle

3. **Jerarquía padre-hijo:**
   - `h3.h3_to_parent(hex_id, res_menor)` — sube de resolución (zoom out)
   - `h3.h3_to_children(hex_id, res_mayor)` — baja de resolución (zoom in)
   - Los hijos de un hexágono res 9 a res 11 son ~49 hexágonos
   - Permite cambiar de escala sin recalcular desde coordenadas

4. **Selección de resolución óptima:**
   - Ciudad: res 7-9, Barrio: res 9-11, Edificio: res 11-13
   - Muchos puntos (millones): res alta (9-11), pocos puntos (miles): res baja (7-8)
   - Visualización ejecutiva: res baja, análisis operativo: res alta

5. **Zoom in/out dinámico:**
   - Mismo dataset, múltiples vistas instantáneas
   - `groupby` por diferentes columnas H3 (h3_res7, h3_res8, h3_res9)
   - Dashboard interactivo: cambiar resolución sin recalcular desde lat/lon

---

### 🎯 Reglas de Oro

👉 **Regla #1: Elegir resolución según escala del análisis**
```python
# MALO: res 11 (~0.002 km²) para analizar una ciudad entera
agg = df.groupby(df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 11), axis=1))['ventas'].sum()
# Resultado: 25,000 hexágonos vacíos o con 1 punto → inútil

# BUENO: res 8 (~0.1 km²) para análisis urbano
df['h3_index'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 8), axis=1)
agg = df.groupby('h3_index')['ventas'].sum()
# Resultado: ~50 hexágonos con datos significativos
```

👉 **Regla #2: Usar h3_to_parent para cambiar resolución sin recalcular**
```python
# MALO: recalcular H3 desde lat/lon para cada resolución
agg_res7 = df.groupby(df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 7), axis=1))['ventas'].sum()
agg_res9 = df.groupby(df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1))['ventas'].sum()

# BUENO: calcular una vez, navegar la jerarquía
df['h3_res9'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)
df['h3_res7'] = df['h3_res9'].apply(lambda x: h3.h3_to_parent(x, 7))
agg_res9 = df.groupby('h3_res9')['ventas'].sum()
agg_res7 = df.groupby('h3_res7')['ventas'].sum()
```

👉 **Regla #3: Validar que cada hexágono tenga suficientes datos**
```python
# MALO: usar resolución alta sin verificar densidad de datos
agg_res11 = df.groupby('h3_res11')['ventas'].mean()
# Muchos hexágonos con 1 registro → promedios no significativos

# BUENO: contar registros por hexágono antes de agregar
conteo = df.groupby('h3_index').size()
hex_validos = conteo[conteo >= 5].index  # mínimo 5 registros
agg = df[df['h3_index'].isin(hex_validos)].groupby('h3_index')['ventas'].mean()
```

---

### 📊 Guía de Decisión

| Situación | Resolución | Método |
|-----------|------------|--------|
| Vista de ciudad completa | 7 (~5 km²) | `h3.geo_to_h3(lat, lon, 7)` |
| Análisis urbano (barrios) | 8 (~0.5 km²) | `h3.geo_to_h3(lat, lon, 8)` |
| Análisis hiper-local | 9 (~0.1 km²) | `h3.geo_to_h3(lat, lon, 9)` |
| Densidad de cuadras | 11 (~0.002 km²) | `h3.geo_to_h3(lat, lon, 11)` |
| Subir de resolución (zoom out) | — | `h3.h3_to_parent(hex_id, res_menor)` |
| Bajar de resolución (zoom in) | — | `h3.h3_to_children(hex_id, res_mayor)` |
| Agregar ventas por hexágono | — | `df.groupby('h3_index')['ventas'].sum()` |
| Contar sucursales por hexágono | — | `df.groupby('h3_index')['sucursal_id'].nunique()` |
| Filtrar hexágonos con datos suficientes | — | `conteo[conteo >= 5].index` |

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔷 ¡Agregación espacial multiescala dominada!</h3>
  <p><i>"La jerarquía H3 permite navegar del continente al edificio sin recalcular: el poder del multiescala."</i></p>
</div>